# Try ReFactX v3

Tests the new `PatternConstrainedGeneration` architecture with:
- **Fact:** triple retrieval with sentinel (`no further records>`)
- **count_branches:** tool for counting KB entries
- **`</think>`** thinking blocks with cache reset

Uses Qwen/Qwen3.5-4B and the v3 prompt.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
import torch
import time
from dotenv import load_dotenv
load_dotenv()

from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoProcessor, AutoModelForImageTextToText, TextStreamer
from transformers import ProcessorMixin

import refactx
from refactx.generate import (
    patch_model, CONSTRAINED_STATES,
    get_constrained_logits_processor,
)

/opt/conda/envs/trl/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/trl/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
MODEL = 'Qwen/Qwen3.5-4B'
INDEX = os.environ.get('POSTGRES_URL', '../indexes/simple_index.txt.gz')
#PROMPT_PATH = '../prompts/prompt_qwen36_mini2_v3.json'
PROMPT_PATH = '/workspace/notebooks/refactx_clean/prompts/prompt_qwen36_angular2.json'

In [4]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cuda'

## Load Model, Index, and Prompt

In [5]:
processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(MODEL, device_map='auto')
tokenizer = processor
streamer = TextStreamer(tokenizer.tokenizer)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 723/723 [00:02<00:00, 298.11it/s]


In [6]:
index = refactx.load_index(INDEX, tokenizer=tokenizer)
print(f'Index type: {type(index).__name__}')

Applying index config...
Index type: PostgresTrieIndex


In [7]:
with open(PROMPT_PATH) as f:
    prompt_messages = json.load(f)
print(f'Prompt loaded from {PROMPT_PATH}')
print(f'System prompt length: {len(prompt_messages[0]["content"])} chars')

Prompt loaded from /workspace/notebooks/refactx_clean/prompts/prompt_qwen36_angular2.json
System prompt length: 3644 chars


In [8]:
patch_model(model)

## Helper Functions

In [9]:
def _tokenize(tok, text):
    if isinstance(tok, ProcessorMixin):
        return tok.tokenizer(text, return_tensors='pt')
    return tok(text, return_tensors='pt')

def _decode(tok, ids):
    if isinstance(tok, ProcessorMixin):
        return tok.tokenizer.decode(ids, skip_special_tokens=True)
    return tok.decode(ids, skip_special_tokens=True)

def ask(question, max_new_tokens=800, sentinel=True):
    """Ask a question using the v3 prompt and PatternConstrainedGeneration."""
    # Build prompt: system prompt + user question
    messages = list(prompt_messages)  # copy
    messages.append({'role': 'user', 'content': question})
    full_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = _tokenize(tokenizer, full_prompt).to(model.device)

    # Fresh constrained processor each call. The shared factory registers the
    # fact and count patterns and forces a multi-token EOT.
    logits_processor = get_constrained_logits_processor(
        tokenizer, index, num_beams=1, num_batches=1,
        sentinel=sentinel)

    model.eval()
    start = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            logits_processor=logits_processor,
            max_new_tokens=max_new_tokens,
            streamer=streamer,
            do_sample=False,
            num_beams=1,
            num_return_sequences=1,
            use_cache=True,
            eos_token_id=tokenizer.tokenizer.eos_token_id
        )
    elapsed = time.time() - start

    state = CONSTRAINED_STATES.states[0][0]
    text = _decode(tokenizer, out[0][inputs.input_ids.shape[1]:])
    facts = state.generated_triples
    facts_str = [_decode(tokenizer, t) for t in facts]

    print(f'\n--- Facts ({len(facts_str)}) ---')
    for i, f in enumerate(facts_str):
        print(f'  {i}: {f}')

    print(f'--- History ({len(state.generation_history)}) ---')
    for i, g in enumerate(state.generation_history):
        cls_name = type(g).__name__
        if hasattr(g, 'completed_with_sentinel'):
            print(f'  [{i}] {cls_name} sentinel={g.completed_with_sentinel}')
        elif hasattr(g, 'called'):
            print(f'  [{i}] {cls_name} called={g.called}')
        else:
            print(f'  [{i}] {cls_name}')

    print(f'Elapsed: {elapsed:.2f}s')
    return text, facts_str

In [10]:
tokenizer.tokenizer.eos_token_id

248046

## Test 1: Multi-hop Reasoning

Ask a question that requires two `Fact:` lookups.

In [11]:
text, facts = ask('When was the director of Slumdog Millionaire born?')

Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


<|im_start|>system
# Fact-Grounded Question Answering

## Role
You are a question-answering system that answers using only verified facts retrieved from a knowledge base.

## Core Principle
All answers must be grounded in retrieved facts. Do not use prior or parametric knowledge when producing answers.

## Reasoning
Reason about how to answer the question. You may use `<think>` blocks for step-by-step reasoning.

Use retrieved facts and counts to update your reasoning and determine what information is needed next.

## Fact Retrieval
When you need information from the knowledge base, call `<fact>` inside a `<think>` block.

The system will return verified knowledge-base facts. You may use multiple `<fact>` calls inside `<think>` blocks when needed.

Only use retrieved facts as evidence for the final answer.

When the system returns `<no further records>`, all objects for that subject-relation pair have been retrieved.

## Counting
When you need to count how many objects exist for a subj

In [12]:
text, facts = ask('who is older? brad pitt or johnny depp?')

Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


<|im_start|>system
# Fact-Grounded Question Answering

## Role
You are a question-answering system that answers using only verified facts retrieved from a knowledge base.

## Core Principle
All answers must be grounded in retrieved facts. Do not use prior or parametric knowledge when producing answers.

## Reasoning
Reason about how to answer the question. You may use `<think>` blocks for step-by-step reasoning.

Use retrieved facts and counts to update your reasoning and determine what information is needed next.

## Fact Retrieval
When you need information from the knowledge base, call `<fact>` inside a `<think>` block.

The system will return verified knowledge-base facts. You may use multiple `<fact>` calls inside `<think>` blocks when needed.

Only use retrieved facts as evidence for the final answer.

When the system returns `<no further records>`, all objects for that subject-relation pair have been retrieved.

## Counting
When you need to count how many objects exist for a subj

## Test 2: Counting

Ask a "how many" question to trigger `count_branches:`.

In [13]:
text, facts = ask('How many countries share a border with France?')

Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


<|im_start|>system
# Fact-Grounded Question Answering

## Role
You are a question-answering system that answers using only verified facts retrieved from a knowledge base.

## Core Principle
All answers must be grounded in retrieved facts. Do not use prior or parametric knowledge when producing answers.

## Reasoning
Reason about how to answer the question. You may use `<think>` blocks for step-by-step reasoning.

Use retrieved facts and counts to update your reasoning and determine what information is needed next.

## Fact Retrieval
When you need information from the knowledge base, call `<fact>` inside a `<think>` block.

The system will return verified knowledge-base facts. You may use multiple `<fact>` calls inside `<think>` blocks when needed.

Only use retrieved facts as evidence for the final answer.

When the system returns `<no further records>`, all objects for that subject-relation pair have been retrieved.

## Counting
When you need to count how many objects exist for a subj

## Test 3: Exhausted Records (Sentinel)

Ask a question that requires enumerating all objects until `no further records>`.

In [14]:
text, facts = ask(
    'Which countries share a border with Brazil?',
    max_new_tokens=1200)

Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


<|im_start|>system
# Fact-Grounded Question Answering

## Role
You are a question-answering system that answers using only verified facts retrieved from a knowledge base.

## Core Principle
All answers must be grounded in retrieved facts. Do not use prior or parametric knowledge when producing answers.

## Reasoning
Reason about how to answer the question. You may use `<think>` blocks for step-by-step reasoning.

Use retrieved facts and counts to update your reasoning and determine what information is needed next.

## Fact Retrieval
When you need information from the knowledge base, call `<fact>` inside a `<think>` block.

The system will return verified knowledge-base facts. You may use multiple `<fact>` calls inside `<think>` blocks when needed.

Only use retrieved facts as evidence for the final answer.

When the system returns `<no further records>`, all objects for that subject-relation pair have been retrieved.

## Counting
When you need to count how many objects exist for a subj

## Test 4: Thinking + Facts

Ask a comparison question that triggers `<think>` reasoning followed by facts.

In [15]:
text, facts = ask(
    'Is Johnny Depp older than Brad Pitt?',
    max_new_tokens=1200)

Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


<|im_start|>system
# Fact-Grounded Question Answering

## Role
You are a question-answering system that answers using only verified facts retrieved from a knowledge base.

## Core Principle
All answers must be grounded in retrieved facts. Do not use prior or parametric knowledge when producing answers.

## Reasoning
Reason about how to answer the question. You may use `<think>` blocks for step-by-step reasoning.

Use retrieved facts and counts to update your reasoning and determine what information is needed next.

## Fact Retrieval
When you need information from the knowledge base, call `<fact>` inside a `<think>` block.

The system will return verified knowledge-base facts. You may use multiple `<fact>` calls inside `<think>` blocks when needed.

Only use retrieved facts as evidence for the final answer.

When the system returns `<no further records>`, all objects for that subject-relation pair have been retrieved.

## Counting
When you need to count how many objects exist for a subj

## Interactive

Edit the question below to test interactively.

In [15]:
MY_QUESTION = 'Who was the first person to walk on the moon?'

text, facts = ask(MY_QUESTION, max_new_tokens=1200)

Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


<|im_start|>system
# Fact-Grounded Question Answering

## Role
You are a question-answering system that answers using only verified facts retrieved from a knowledge base.

## Core Principle
All answers must be grounded in retrieved facts. Do not use prior or parametric knowledge when producing answers.

## Reasoning
Reason about how to answer the question. You may use `<think>` blocks for step-by-step reasoning.

Use retrieved facts and counts to update your reasoning and determine what information is needed next.

## Fact Retrieval
When you need information from the knowledge base, call `<fact>` inside a `<think>` block. The retrieval command must be part of the thinking process, not emitted as a standalone block.

The system will return verified knowledge-base facts. You may use multiple `<fact>` calls inside `<think>` blocks when needed.

Only use retrieved facts as evidence for the final answer.

When the system returns `<no further records>`, all objects for that subject-relation 